# Example 2: BSCAN-controlled adders, simplex against TMR, single-bit upsets

This example programs a bitstream. Then it compares a simplex adder with a triplicated (TMR) adder. Each adder is in its own pblock.

The two adders use one operand register (USER1). Each adder has its own result register: USER2 for the simplex adder and USER3 for the TMR adder.

![Block diagram of bscan_adder_top. BSCAN USER1 sends the operands to a simplex adder and to three adders with a voter (TMR). Each group is in its own pblock. USER2 and USER3 return the results over JTAG.](images/example2_bscan_adder_top_abstract.svg)

Each trial does these steps:

1. It injects one random essential bit into each pblock.
2. It sends 100 random operand pairs through the two adders.
3. It marks an adder as failed if one or more of its results are wrong.
4. It flips the two bits again to correct the upsets.

## The calls of the example

| Call | Use in the example |
| --- | --- |
| `SkraftFpga.connect(ip=..., port=...)`, `client.attach(SERIAL_NUMBER)` | Connect to the SK-RAFT server and attach to the device, in `with` statements that release the device and close the connection. |
| `Design(bit, ebd, xdc)`, `dev.uploadDesignToServer(design)`, `dev.programFPGA(design)` | Give the design files, upload them to the server and program the FPGA. |
| `dev.configure_seed(0)` | Set the seed of the random injections. The same seed gives the same upsets in each run. |
| `BscanPrimitive(user, length)`, `dev.write_bscan(primitive, value)`, `dev.read_bscan(primitive)` | Write the operands through USER1, and read the results of the simplex adder (USER2) and of the TMR adder (USER3). |
| `dev.bit_count(TargetMemory.ESSENTIAL_CRAM, pblock=...)` | Get the number of essential bits of each adder pblock. |
| `dev.inject_random(TargetMemory.ESSENTIAL_CRAM, pblock=...)` | Inject one upset into a random essential bit of each adder pblock. |
| `dev.inject_at(far, word, bit, TargetMemory.CRAM)` | Flip the two bits again after the trial, to correct the upsets. |

| Constant | Default | Description |
| --- | --- | --- |
| `design` | `bscan_adder_top.{bit,ebd}` + XDC | The design files. The script uploads them to the server. |
| `PBLOCK_SIMPLEX` / `PBLOCK_TMR` | `"pblock_sngl_add"` / `"pblock_tmr_add"` | The two pblocks for the test, as the XDC file names them. |
| `OP_LEN` | `4` | The operand width. It must be equal to the `op_len` generic of the design. |

## Result

Sample output of a run on a Basys 3 (shortened). The frame addresses, the bit counts and the error counts depend on the board, the build and the seed.

```
Programmed 210183BFD452A.
Number of Essential Bits in:
Simplex Adder = 595 | TMR Adder = 3374
Press Enter to move on.
--------------------------------------------------------------------------------
Vulnerability Factor (VF) = observable errors / injected faults
# of trials = 156  ||  Simplex Correct VF = 66/156 (42.31%)  |  TMR Correct VF = 5/156 (3.21%)
66 simplex, 5 tmr errors in 156 trials.
Final Simplex Adder VF = 66/156 (42.31%)
Final TMR Adder VF     = 5/156 (3.21%)
Example 2 done.
```

First, the script prints the number of essential bits in each pblock. Then it updates one status line during the trials. The status line shows:

- The number of trials.
- The result of each adder for the last trial: green "Correct" or red "Wrong".
- The vulnerability factor (VF) of each adder: the fraction of trials in which the adder failed.

When you stop the script, it prints the final error counts and the VFs.

Each trial corrects its bits immediately after the check. Thus the configuration of the device is clean at the end. If you stop the script during a trial, a bit can stay flipped. In that case, program the FPGA again before you do other work.

## Run the example

### 0: Install the module

This will restart the kernel, a popup will show up, just ignore it.

In [1]:
%pip install ../../api/python

import IPython; IPython.get_ipython().kernel.do_shutdown(restart=True)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Defaulting to user installation because normal site-packages is not writeable
Processing /home/wallmenis/git/sk-raft-client/api/python
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for skraft: filename=skraft-1.0.0-py3-none-any.whl size=45974 sha256=2974e2a881bab82f907bd88f2a8ac3cc868e632a60c3405f303fea49003e40e5
  Stored in directory: /tmp/pip-ephem-wheel-cache-dio5xx2m/wheels/cc/7e/0f/603cd9eaf8e594f1c766c921daeeef2440d17e0cbb287bfb57
Successfully built skraft
  Attempting uninstall: skraft
    Found existing installation: skraft 1.0.0
    Uninstalling skraft-1.0.0:
      Successfully uninstalled skraft-1.0.0
Note: you may need to restart the kernel to use updated packages.


{'status': 'ok', 'restart': True}

### 1: Extract the design and bitstream files

On Linux

In [1]:
%%bash
cd ../hw/design_examples/
./unzip_designs.sh

Design files extracted to /home/wallmenis/git/sk-raft-client/get_started/hw/design_examples


On Windows

In [ ]:
%%script pwsh
cd ..\hw\design_examples\
.\unzip_designs.bat

### 2: Add the imports

In [6]:
import os
import random
import threading
import time

from skraft import Design, TargetMemory, ReadbackOption, SkraftError, SkraftFpga, BscanPrimitive

SK_RAFT_SERVER_IP = "127.0.0.1"
SK_RAFT_SERVER_PORT = 50051

### 3: Start the server

In [7]:
%%script bash --bg
sk-raft

### 4: List devices
Connect the devices you want to test before running list devices.

In [8]:
def main() -> int:
    try:
        # SK-RAFT: open the connection. Then list the connected boards.
        with SkraftFpga.connect(ip=SK_RAFT_SERVER_IP,
                                port=SK_RAFT_SERVER_PORT) as client:
            devices = client.get_devices()
    except SkraftError as e:
        print(f"Cannot list the boards: {e}")
        return 1
    except KeyboardInterrupt:
        print("Interrupted.")
        return 130

    if not devices:
        print("No board is connected to the SK-RAFT server.")
        return 1
    print(f"{'SERIAL NUMBER':16s} PART NUMBER")
    for device in devices:
        print(f"{device.serial_number:16s} {device.part_number}")
    return 0


if __name__ == "__main__":
    main()

SERIAL NUMBER    PART NUMBER
210183BA2524A    XC7A35T


### 5: Setup the code
Don't forget to change the Serial Number (`SERIAL_NUMBER`) and Board (`BOARD`) to the ones you desire from the list!

In [12]:
SERIAL_NUMBER = "210183BA2524A"

# The design for all trials: a simplex adder and a TMR adder.
# The script gives all three files at the start:
#   The bitstream (.bit): to program the device.
#   The EBD file (.ebd): to find which bits are essential.
#   The XDC file (.xdc): to find the pblocks.
# uploadDesignToServer() sends the files to the server. The client sends
# only the file name of each path, not the directory.
#
# BOARD is the board of your target device: "basys3" or "zedboard", or
# the name of your own board (see hw/README.md). The script reads the
# design files from hw/design_examples/<BOARD>/bscan_adder_top/. Extract
# design_examples.zip there, or build the design with create_project.tcl.
# The paths are relative to src/. Run the script from that folder.
BOARD = "basys3"
DESIGN_DIR = f"../hw/design_examples/{BOARD}/bscan_adder_top"

design = Design(
    bit=f"{DESIGN_DIR}/bscan_adder_top.bit",
    ebd=f"{DESIGN_DIR}/bscan_adder_top.ebd",
    xdc=f"{DESIGN_DIR}/board.xdc",
)

# The two pblocks to test, with the names from the XDC file.
PBLOCK_SIMPLEX = "pblock_sngl_add"
PBLOCK_TMR = "pblock_tmr_add"

# The script corrects the upsets of each trial immediately after the check
# (non-accumulative). The trials continue until you press Enter.

# The op_len generic of the design. OP_LEN sets the width of USER1 and
# of USER2/USER3 (with the carry bit). If you build the design again
# with a different value, set OP_LEN to the same value.
OP_LEN = 4

# The script packs each operand pair as (operand B << OP_LEN) | operand A.
PAIR_LEN = OP_LEN * 2

RESULT_MASK = (1 << (OP_LEN + 1)) - 1
OP_MASK = (1 << OP_LEN) - 1

# BSCAN registers: (USER number, width in bits)
USER_OPERANDS = BscanPrimitive(1, PAIR_LEN)
USER_SIMPLEX_RESULT = BscanPrimitive(2, OP_LEN + 1)
USER_TMR_RESULT = BscanPrimitive(3, OP_LEN + 1)


# Green/red pass/fail labels. os.system("") makes the classic Windows
# console show ANSI colours. Other terminals ignore this call.
os.system("")
GREEN, RED, RESET = "\033[32m", "\033[31m", "\033[0m"

### 6: Define some helper functions

In [13]:
def verdict(ok: bool) -> str:
    return f"{GREEN}Correct{RESET}" if ok else f"{RED}Wrong{RESET}  "


# Write one operand pair to both adders (USER1).
def set_operands(dev, a: int, b: int) -> None:
    # SK-RAFT: write the packed operand pair.
    dev.write_bscan(USER_OPERANDS, (b << OP_LEN) | a)


# Read the result of the simplex adder (USER2).
def read_simplex_result(dev) -> int:
    # SK-RAFT: read the result. Then unpack it.
    data = dev.read_bscan(USER_SIMPLEX_RESULT)
    return int.from_bytes(data, "big") & RESULT_MASK


# Read the result of the TMR adder (USER3).
def read_tmr_result(dev) -> int:
    # SK-RAFT: read the result. Then unpack it.
    data = dev.read_bscan(USER_TMR_RESULT)
    return int.from_bytes(data, "big") & RESULT_MASK

### 7: Connect to the server

In [14]:
# A background thread monitors the Enter key. Thus, the injection loop
# below does not wait for input. Press Enter at any time to stop the
# current step early.
client = SkraftFpga.connect(ip=SK_RAFT_SERVER_IP, port=SK_RAFT_SERVER_PORT)
dev = client.attach(SERIAL_NUMBER)

### 8: Program the board...
...and upload the design to the server because it needs it for some operations.

In [15]:
dev.uploadDesignToServer(design)
dev.programFPGA(design)
print(f"programmed {SERIAL_NUMBER} with {design.bit}")

programmed 210183BA2524A with ../hw/design_examples/basys3/bscan_adder_top/bscan_adder_top.bit


### 9: Run the experiment!
The `N_OF_INJECTIONS` is the number of fault injections the experiment will do.

In [18]:

N_OF_INJECTIONS = 100


# Setup the SEED for deterministic behaviour. 
dev.configure_seed(0)

simplex_ebits = dev.bit_count(TargetMemory.ESSENTIAL_CRAM, pblock=PBLOCK_SIMPLEX)
                    
tmr_ebits = dev.bit_count(TargetMemory.ESSENTIAL_CRAM, pblock=PBLOCK_TMR)
                                          
print("Number of Essential Bits in:")
print(f"Simplex Adder = {simplex_ebits} | TMR Adder = {tmr_ebits}")
print("-" * 80)
print("Vulnerability Factor (VF) = observable errors / injected faults")
simplex_error = 0
tmr_error = 0
iterations = N_OF_INJECTIONS
n = 0
while iterations!=0:
    simplex_ok = True
    tmr_ok = True
    # Inject one random upset into an essential bit of each
    # adder.
    report_sim = dev.inject_random(TargetMemory.ESSENTIAL_CRAM, pblock=PBLOCK_SIMPLEX)
    report_tmr = dev.inject_random(TargetMemory.ESSENTIAL_CRAM, pblock=PBLOCK_TMR)
    # Apply 100 random operand pairs with the same upset in
    # place. This number is enough to test both circuits. If
    # one or more operand pairs give an error, add 1 to the
    # error count of the adder that failed.
    for _ in range(100):
        a = random.randint(0, OP_MASK)
        b = random.randint(0, OP_MASK)
        # Calculate the expected result.
        expected = (a + b) & RESULT_MASK
        # Write the operands to BSCAN register USER1.
        set_operands(dev, a, b)
        # Read the result of each adder.
        simplex_result = read_simplex_result(dev)
        tmr_result = read_tmr_result(dev)
        # Set the pass/fail status of each adder. One wrong
        # result is sufficient to mark the adder as failed for
        # this trial.
        if simplex_result != expected:
            simplex_ok = False
        if tmr_result != expected:
            tmr_ok = False
    # Add 1 to the error count of each adder that failed. The
    # count grows by 1 for each trial at most. Thus, VF is a
    # fraction of trials, not of operand pairs.
    if not simplex_ok:
        simplex_error += 1
    if not tmr_ok:
        tmr_error += 1
    # Correct the upset in both adders. Essential bits are
    # CRAM bits. Thus, use the plain CRAM target to flip
    # them again.
    # Set to False to keep the injected upsets
    # (accumulative fault injection).
    if True:
        dev.inject_at(report_sim.far, report_sim.word_index, report_sim.bit_index, TargetMemory.CRAM)
        dev.inject_at(report_tmr.far, report_tmr.word_index, report_tmr.bit_index, TargetMemory.CRAM)
    n += 1
    iterations -= 1
    # After each trial, the script writes the status line again
    # over the same line (\r). The line shows the result of
    # each adder for this trial (green/red) and the current
    # VFs.
    print(f"\r# of trials = {n}  ||  "
    f"Simplex {verdict(simplex_ok)} "
    f"VF = {simplex_error}/{n} "
    f"({simplex_error/n * 100:.2f}%)  |  "
    f"TMR {verdict(tmr_ok)} "
    f"VF = {tmr_error}/{n} "
    f"({tmr_error/n * 100:.2f}%)   ",
    end="", flush=True)
if n == 0:
    print("\nNo trials completed.")
else:
    print(f"\n{simplex_error} simplex, {tmr_error} tmr errors in {n} trials.")
    print(f"Final Simplex Adder VF = {simplex_error}/{n} ({simplex_error/n * 100:.2f}%)")
    print(f"Final TMR Adder VF     = {tmr_error}/{n} ({tmr_error/n * 100:.2f}%)")

Number of Essential Bits in:
Simplex Adder = 595 | TMR Adder = 3374
--------------------------------------------------------------------------------
Vulnerability Factor (VF) = observable errors / injected faults
# of trials = 100  ||  Simplex Correct VF = 46/100 (46.00%)  |  TMR Correct VF = 2/100 (2.00%)   
46 simplex, 2 tmr errors in 100 trials.
Final Simplex Adder VF = 46/100 (46.00%)
Final TMR Adder VF     = 2/100 (2.00%)
